# Diagnostics — does the frozen embedding see what the annotators saw?

The training notebook asks *is the fusion better than the baselines?*. This one asks a different
question: **do the frozen features contain the cues a human coder annotated?**

`dataset.csv` ships 41 MUMIN annotations per clip (36 visual + 5 vocal, see
[dataset/README.md](../../../dataset/README.md#7-the-41-mumin-annotations)). They are made by
hand, so they can never be an input at inference time — but they are a reference: if a linear
probe can read `Gaze Up` or `Head Tilt` out of the DINOv2 embedding, the information is there and
the problem is in the head or the fusion. If it cannot, the problem is the representation — 16
frames, mean pooling, face crop — and no fusion module will fix it.

**This notebook produces no board row.** It produces a diagnosis: which cues are linearly
accessible from each stream, and which are simply not there.

## 0 — Setup

In [10]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# CODE_ROOT is the submitted `code/` folder (it holds lib/ and scripts/). Upload it to Drive and
# adjust the default below, or set the CODE_ROOT environment variable before running.
import os, sys
CODE_ROOT = os.environ.get("CODE_ROOT", "/content/drive/MyDrive/code")
%cd {CODE_ROOT}
sys.path.insert(0, '.')


## 1 — Settings

In [ ]:
from pathlib import Path

from lib.dataset import cache_dir

DATA = Path(os.environ.get("DOLOS_DATA", "/content/drive/MyDrive/dolos"))
VIDEO_CACHE = cache_dir("dinov2-small", DATA)
AUDIO_CACHE = cache_dir("wavlm-base-plus", DATA)

READ_WORKERS = 16        # the Drive mount is latency-bound: overlap the reads
MIN_PREVALENCE = 0.05    # skip features that appear in fewer clips: their AUC is not estimable
NULL_TARGETS = 10        # random targets per prevalence level, for the noise band
C_REG = 0.1              # strong regularisation: ~900 clips, 384-768 dimensions
SEED = 0

print(VIDEO_CACHE)
print(AUDIO_CACHE)

/content/drive/MyDrive/dolos/cached_features/dinov2-small
/content/drive/MyDrive/dolos/cached_features/wavlm-base-plus


## 2 — Protocol and clips

Same frozen protocol as the training notebook: the probe is trained on a fold's `train` and scored
on its `holdout`, so no subject is ever on both sides.

In [13]:
import numpy as np
import pandas as pd

from lib.dataset import Clips, Protocol

folds = Protocol.load(DATA / "splits")
clips = Clips.load(DATA)
print(clips.describe_source())

ids = []
for fold in range(1, folds.folds + 1):
    for part in folds.subject(fold):
        ids += part
ids = list(dict.fromkeys(ids))
ids = [c for c in clips.keep(ids)
       if (VIDEO_CACHE / f"{c}.npy").exists() and (AUDIO_CACHE / f"{c}.npy").exists()]
print(f"{len(ids)} working clips with both arrays")

dataset.csv sha256:8fa198b1fe71  1678 clips  1452 with media
1158 working clips with both arrays


## 3 — Pooled embeddings, once

The mean over time is exactly what `LateFusion` sees, so this measures the representation the
model actually has — not a better one.

In [14]:
from concurrent.futures import ThreadPoolExecutor


def pooled(cache, label):
    def read(clip_id):
        array = np.load(cache / f"{clip_id}.npy").astype(np.float32)
        return clip_id, (array if array.ndim == 1 else array.mean(axis=0))

    out = {}
    with ThreadPoolExecutor(READ_WORKERS) as pool:
        for clip_id, vector in pool.map(read, ids):
            out[clip_id] = vector
    print(f"{label}: {len(out)} vectors of {len(next(iter(out.values())))}")
    return out


video = pooled(VIDEO_CACHE, "video")
audio = pooled(AUDIO_CACHE, "audio")
x_video = np.stack([video[c] for c in ids])
x_audio = np.stack([audio[c] for c in ids])
row = {clip_id: i for i, clip_id in enumerate(ids)}

video: 1158 vectors of 384
audio: 1158 vectors of 768


## 4 — The annotations

24 of the 41 features appear in fewer than 10% of the clips, so a probe on them would be measuring
noise. Everything below `MIN_PREVALENCE` is skipped.

In [15]:
meta = pd.read_csv(DATA / "dataset.csv", dtype={"clip_id": str}).set_index("clip_id")
mumin_columns = [c for c in meta.columns if c.startswith("mumin_")]
mumin = (meta.loc[ids, mumin_columns]
         .apply(pd.to_numeric, errors="coerce").fillna(0).to_numpy(np.float32))
prevalence = mumin.mean(axis=0)
probe_columns = [j for j, p in enumerate(prevalence) if p >= MIN_PREVALENCE]

print(f"{len(mumin_columns)} MUMIN columns, {len(probe_columns)} above {MIN_PREVALENCE:.0%}")
for j in probe_columns:
    print(f"  {mumin_columns[j].replace('mumin_', ''):28s} {prevalence[j]:5.1%}")

41 MUMIN columns, 26 above 5%
  smile                        13.5%
  laughter                      5.2%
  frowning                     13.3%
  raising                      46.5%
  frequent_blinking            18.4%
  corners_up                   20.5%
  corners_down                  8.5%
  protruded                     5.6%
  move_upwards                 17.7%
  move_downwards               14.8%
  towards_interlocutor         78.2%
  up                           12.3%
  down                         29.4%
  sideways                     23.7%
  nod                          25.9%
  move_forward                  8.6%
  move_backward                 9.1%
  tilt                         23.9%
  side_turn                    23.2%
  shake                        11.0%
  head_up                       6.6%
  touching_head_body            5.3%
  palm_up                      12.6%
  open                         17.4%
  non_ah_mmh_disturbances       7.6%
  word_phrase_repetitions       8.7%


## 5 — The probe and the noise band

A logistic regression from the pooled embedding to one annotation, trained on `train` and scored on
`holdout`.

With 26 features and five folds, some AUCs will be high by chance — and the chance level depends on
how rare the annotation is: ranking 60 positives against 1,100 negatives is noisier than ranking 350
against 800. So the null is measured **at several prevalence levels** with random targets, and each
feature is compared against the band interpolated at its own prevalence. The band is the null mean
plus 1.645 standard deviations: one-sided 95%.

In [16]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm


def rows(clip_ids):
    return np.array([row[c] for c in clip_ids if c in row], dtype=int)


def fold_auc(x, target):
    """AUC on each fold's holdout, training on its train split: subject-disjoint."""
    scores = []
    for fold in range(1, folds.folds + 1):
        train_ids, _, hold_ids = folds.subject(fold)
        train, hold = rows(train_ids), rows(hold_ids)
        if len(np.unique(target[train])) < 2 or len(np.unique(target[hold])) < 2:
            continue
        clf = make_pipeline(StandardScaler(),
                            LogisticRegression(max_iter=1000, C=C_REG, class_weight="balanced"))
        clf.fit(x[train], target[train])
        scores.append(roc_auc_score(target[hold], clf.predict_proba(x[hold])[:, 1]))
    return np.array(scores)


# A rare annotation is harder to rank than a common one, so the noise band has to follow the
# prevalence: random targets are drawn at a few prevalence levels and interpolated per feature.
PREVALENCE_GRID = (0.05, 0.10, 0.20, 0.30, 0.40, 0.50)
rng = np.random.default_rng(SEED)
bands = {}
for level in tqdm(PREVALENCE_GRID, desc="noise band", unit="level"):
    scores = {"video": [], "audio": []}
    for _ in range(NULL_TARGETS):
        target = rng.binomial(1, level, size=len(ids)).astype(np.float32)
        scores["video"].append(fold_auc(x_video, target).mean())
        scores["audio"].append(fold_auc(x_audio, target).mean())
    bands[level] = {name: float(np.mean(v) + 1.645 * np.std(v)) for name, v in scores.items()}
    print(f"  prevalence {level:.0%}: band video {bands[level]['video']:.3f}  "
          f"audio {bands[level]['audio']:.3f}")


def band_at(feature_prevalence, modality):
    """The band for a feature, interpolated between the prevalence levels measured above."""
    levels = np.array(sorted(bands))
    values = np.array([bands[level][modality] for level in levels])
    return float(np.interp(feature_prevalence, levels, values))

noise band:   0%|          | 0/6 [00:00<?, ?level/s]

  prevalence 5%: band video 0.609  audio 0.551
  prevalence 10%: band video 0.561  audio 0.543
  prevalence 20%: band video 0.557  audio 0.528
  prevalence 30%: band video 0.557  audio 0.540
  prevalence 40%: band video 0.548  audio 0.551
  prevalence 50%: band video 0.539  audio 0.533


## 6 — Results

In [17]:
from pathlib import Path

results = []
for j in tqdm(probe_columns, desc="MUMIN", unit="feat"):
    target = (mumin[:, j] > 0).astype(np.float32)
    v, a = fold_auc(x_video, target), fold_auc(x_audio, target)
    band_video, band_audio = band_at(prevalence[j], "video"), band_at(prevalence[j], "audio")
    results.append({"feature": mumin_columns[j].replace("mumin_", ""),
                    "prevalence": prevalence[j],
                    "video": v.mean(), "band_video": band_video,
                    "audio": a.mean(), "band_audio": band_audio,
                    "above_video": bool(v.mean() > band_video),
                    "above_audio": bool(a.mean() > band_audio)})

table = pd.DataFrame(results)
table["best"] = table[["video", "audio"]].max(axis=1)
table["captured"] = np.where(table["above_video"] | table["above_audio"], "yes", "no")
table = table.sort_values("best", ascending=False).reset_index(drop=True)
print(table[["feature", "prevalence", "video", "band_video", "audio", "band_audio",
             "captured"]].round(3).to_string(index=False))
print()
print(f"above its own band: {int((table['captured'] == 'yes').sum())} of {len(table)}  |  "
      f"video {int(table['above_video'].sum())}  audio {int(table['above_audio'].sum())}")

out = Path("results"); out.mkdir(exist_ok=True)
table.to_csv(out / "diagnostics_mumin.csv", index=False)
print("saved", out / "diagnostics_mumin.csv")

MUMIN:   0%|          | 0/26 [00:00<?, ?feat/s]

                feature  prevalence  video  band_video  audio  band_audio captured
non_ah_mmh_disturbances       0.076  0.692       0.584  0.818       0.547      yes
     touching_head_body       0.053  0.759       0.606  0.574       0.550      yes
           corners_down       0.085  0.756       0.575  0.666       0.545      yes
   towards_interlocutor       0.782  0.753       0.539  0.644       0.533      yes
               laughter       0.052  0.707       0.607  0.737       0.550      yes
                  smile       0.135  0.736       0.559  0.601       0.538      yes
                   open       0.174  0.733       0.558  0.611       0.532      yes
                   down       0.294  0.730       0.557  0.629       0.539      yes
                head_up       0.066  0.730       0.594  0.584       0.548      yes
              protruded       0.056  0.715       0.603  0.602       0.550      yes
             corners_up       0.205  0.713       0.557  0.585       0.528      yes
    

In [18]:
# one line per stream: how much of the annotated behaviour is linearly readable at all
print(f"mean AUC over features   video {table['video'].mean():.3f}   audio {table['audio'].mean():.3f}")
print(f"mean band for these       video {table['band_video'].mean():.3f}   "
      f"audio {table['band_audio'].mean():.3f}")
print(f"margin over the band     video {(table['video'] - table['band_video']).mean():+.3f}   "
      f"audio {(table['audio'] - table['band_audio']).mean():+.3f}")

mean AUC over features   video 0.665   audio 0.585
mean band for these       video 0.567   audio 0.539
margin over the band     video +0.098   audio +0.046


## 7 — Reference: the annotations alone

The probes above ask *can the embedding read a cue*. This asks the other half: **if you had the
annotations, how well would deception be predictable?** It is the paper's Table 4 baseline
(58.84% ACC on its own protocol), reproduced here on our subject-level folds.

It is a **reference, not a model**: the 41 columns are filled by a human watching the clip, so at
inference time they do not exist. Read it as the ceiling the cues can reach. If it lands near 60%
ACC, then the fusion at 56.5 is close to what the annotations themselves support, and the gap to
the paper is the protocol, not the model.

In [19]:
from pathlib import Path

from lib.metrics import best_threshold, evaluate

y = clips.labels(ids).astype(int)
oracle, oracle_preds = [], []
for fold in range(1, folds.folds + 1):
    train_ids, val_ids, hold_ids = folds.subject(fold)
    train, val, hold = rows(train_ids), rows(val_ids), rows(hold_ids)
    clf = make_pipeline(StandardScaler(),
                        LogisticRegression(max_iter=2000, C=0.1, class_weight="balanced"))
    clf.fit(mumin[train], y[train])
    threshold = best_threshold(y[val], clf.predict_proba(mumin[val])[:, 1])
    prob = clf.predict_proba(mumin[hold])[:, 1]
    metrics = evaluate(y[hold], prob, threshold=threshold)
    oracle.append(metrics)
    oracle_preds.append(pd.DataFrame({"clip_id": [ids[i] for i in hold],
                                      "y_true": y[hold], "y_prob": prob}))
    print(f"fold {fold}: ACC {metrics['acc']*100:5.1f}  AUC {metrics['auc']*100:5.1f}")

frame = pd.DataFrame(oracle)
print(f"\nORACLE  ACC {frame.acc.mean()*100:.1f} ± {frame.acc.std()*100:.1f}  "
      f"AUC {frame.auc.mean()*100:.1f}  balanced {frame.balanced_acc.mean()*100:.1f}")

out = Path("results"); out.mkdir(exist_ok=True)
pd.concat(oracle_preds, ignore_index=True).to_csv(out / "oracle_mumin.csv", index=False)
print("saved", out / "oracle_mumin.csv")

fold 1: ACC  53.1  AUC  57.7
fold 2: ACC  58.0  AUC  54.7
fold 3: ACC  50.6  AUC  54.3
fold 4: ACC  49.7  AUC  54.0
fold 5: ACC  44.1  AUC  41.9

ORACLE  ACC 51.1 ± 5.1  AUC 52.5  balanced 51.4
saved results/oracle_mumin.csv


## 8 — How to read it

- **A feature above the noise band** is *linearly accessible* from that stream: the information
  survives the crop, the 16 frames and the mean pooling. It does not mean the cue helps deception —
  only that the embedding carries it.
- **A feature at the band** is not there. If the visual features a human coder needed (`Gaze Up`,
  `Head Tilt`, `Frequent Blinking`) are all at the band, the video branch is not under-trained, it is
  under-informed: the fix is more frames, a different crop or a temporal pooling, not a bigger
  fusion module.
- **`Blinking`, `Silent pauses`, `Word repetitions`** are the interesting ones: they are temporal by
  nature, and a mean over 16 samples destroys them. An AUC at the band there is expected and it is
  the argument for delta features or attention pooling.
- The probe is **linear**. A non-linear probe would find more, so "not captured" means "not
  linearly accessible", not "absent". With ~900 clips and 384-768 dimensions, that is the strongest
  claim the data supports.

- **The band depends on the sample size**, not on the features: with 162 holdout clips per fold it
  lands around 0.54, so an AUC of 0.58 is real but modest. If the band comes out near 0.65, the
  probe has no power on this data and the honest conclusion is "not measurable", not "not captured".
- To compare two representations, change `VIDEO_CACHE` — for instance to `dinov2-small-body` once
  that cache exists — and run the same cell: the question becomes *does the wider crop make the
  coded behaviours more readable?*.
- **Compare the two halves.** If most cues are readable but the oracle sits near 60% ACC, the cues
  are weakly tied to the label and the ceiling is the dataset, not the architecture. If the oracle
  were much higher, the representation would be leaving signal on the table and the model would be
  the place to work.